In [4]:
# FILE PATHS
import re
import pandas as pd
from pathlib import Path

INDUSTRY_FILE = Path(
    r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\master_jobs_industry_improved.csv"
)

LLM_FILE = Path(
    r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\master_jobs_llm.csv"
)

FINAL_FILE = Path(
    r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\final_labour_market_dataset.csv"
)


# LOAD BOTH DATASETS
industry_df = pd.read_csv(INDUSTRY_FILE)
llm_df = pd.read_csv(LLM_FILE)

print("=" * 70)
print("DATASETS LOADED")
print("=" * 70)

print("\nIndustry dataset:")
print("Rows:", len(industry_df))
print("Columns:", industry_df.columns.tolist())

print("\nLLM dataset:")
print("Rows:", len(llm_df))
print("Columns:", llm_df.columns.tolist())

DATASETS LOADED

Industry dataset:
Rows: 19207
Columns: ['job_id', 'source_id', 'title', 'company', 'location', 'description', 'salary_min', 'salary_max', 'salary_avg', 'created', 'contract_type', 'contract_time', 'url', 'source', 'title_clean', 'description_clean', 'industry', 'industry_original', 'second_pass_industry', 'industry_before_second_pass']

LLM dataset:
Rows: 19207
Columns: ['_row_index', 'job_id', 'title', 'industry', 'status', 'technical_skills', 'soft_skills', 'tools_and_technologies', 'certifications', 'licenses', 'education_required', 'experience_required', 'languages']


In [5]:
# JOB ID CHECK
print("\n" + "=" * 70)
print("JOB ID CHECK")
print("=" * 70)

print(
    "Industry duplicate job IDs:",
    industry_df["job_id"].duplicated().sum()
)

print(
    "LLM duplicate job IDs:",
    llm_df["job_id"].duplicated().sum()
)

industry_ids = set(industry_df["job_id"])
llm_ids = set(llm_df["job_id"])

print(
    "\nJobs in industry dataset:",
    len(industry_ids)
)

print(
    "Jobs in LLM dataset:",
    len(llm_ids)
)

print(
    "Common job IDs:",
    len(industry_ids & llm_ids)
)

print(
    "Only in industry dataset:",
    len(industry_ids - llm_ids)
)

print(
    "Only in LLM dataset:",
    len(llm_ids - industry_ids)
)


JOB ID CHECK
Industry duplicate job IDs: 0
LLM duplicate job IDs: 0

Jobs in industry dataset: 19207
Jobs in LLM dataset: 19207
Common job IDs: 19207
Only in industry dataset: 0
Only in LLM dataset: 0


In [6]:
# MERGE INDUSTRY + LLM EXTRACTION
# Keep only the LLM-extracted columns we need
llm_columns = [
    "job_id",
    "technical_skills",
    "soft_skills",
    "tools_and_technologies",
    "certifications",
    "licenses",
    "education_required",
    "experience_required",
    "languages",
    "status"
]

llm_features = llm_df[llm_columns].copy()

# Remove industry from LLM dataset because we use
# the improved hybrid industry classification
# from industry_df.

final_df = industry_df.merge(
    llm_features,
    on="job_id",
    how="inner",
    validate="one_to_one",
    suffixes=("", "_llm")
)

print("=" * 70)
print("DATASETS MERGED")
print("=" * 70)

print("\nRows:", len(final_df))
print("\nColumns:")
print(final_df.columns.tolist())

DATASETS MERGED

Rows: 19207

Columns:
['job_id', 'source_id', 'title', 'company', 'location', 'description', 'salary_min', 'salary_max', 'salary_avg', 'created', 'contract_type', 'contract_time', 'url', 'source', 'title_clean', 'description_clean', 'industry', 'industry_original', 'second_pass_industry', 'industry_before_second_pass', 'technical_skills', 'soft_skills', 'tools_and_technologies', 'certifications', 'licenses', 'education_required', 'experience_required', 'languages', 'status']


In [7]:
# FINAL MERGE QUALITY CHECK
print("=" * 70)
print("FINAL MERGE QUALITY CHECK")
print("=" * 70)

print("\nOriginal industry dataset:", len(industry_df))
print("Original LLM dataset:", len(llm_df))
print("Final merged dataset:", len(final_df))

print(
    "\nDuplicate job IDs:",
    final_df["job_id"].duplicated().sum()
)

print(
    "Missing job IDs:",
    final_df["job_id"].isna().sum()
)

print(
    "\nIndustry missing:",
    final_df["industry"].isna().sum()
)

print(
    "Technical skills missing:",
    final_df["technical_skills"].isna().sum()
)

print(
    "Soft skills missing:",
    final_df["soft_skills"].isna().sum()
)

print(
    "Tools missing:",
    final_df["tools_and_technologies"].isna().sum()
)

print(
    "Experience missing:",
    final_df["experience_required"].isna().sum()
)

FINAL MERGE QUALITY CHECK

Original industry dataset: 19207
Original LLM dataset: 19207
Final merged dataset: 19207

Duplicate job IDs: 0
Missing job IDs: 0

Industry missing: 0
Technical skills missing: 0
Soft skills missing: 0
Tools missing: 0
Experience missing: 10365


In [8]:
# FINAL DATASET PREVIEW
display(
    final_df[
        [
            "job_id",
            "title",
            "company",
            "location",
            "industry",
            "technical_skills",
            "soft_skills",
            "tools_and_technologies",
            "certifications",
            "licenses",
            "education_required",
            "experience_required",
            "languages",
            "salary_avg",
            "contract_type",
            "contract_time"
        ]
    ].head(10)
)

,job_id,title,company,location,industry,technical_skills,soft_skills,tools_and_technologies,certifications,licenses,education_required,experience_required,languages,salary_avg,contract_type,contract_time
0,0d6b3088-b9b3-4238-8be0-9ef2704750ac,Dispenser NVQ,Superdrug,PR81DJ,Healthcare,[],[],[],['NVQ'],[],NaN,Qualified dispenser pharmacy experience,[],13.40,Unknown,Unknown
1,58cb2c2b-6f47-4ffa-aec2-dde531278d24,Service Colleague Express,ASDA,G659AE,Retail,[],[],[],[],[],NaN,NaN,[],13.10,Unknown,Unknown
2,ba4c3cc1-dde8-4122-9f64-5edfd3297e11,Service Colleague Express,ASDA,BA114NZ,Retail,[],[],[],[],[],NaN,NaN,[],13.10,Unknown,Unknown
3,f8ec9f92-d35a-4058-b376-8b43222d0e50,Customer Delivery Driver,ASDA,SK11UA,Transport and Logistics,[],[],[],[],[],NaN,2 years of experience,[],13.10,Unknown,Unknown
4,4b4bfd85-4575-409f-8f48-25a3bb4cdc7b,IT Security Technical Delivery Manager VP,eFinancialCareers,London,IT and Technology,[],[],[],[],[],NaN,NaN,[],NaN,Unknown,Unknown
5,2ed5c775-f8f2-4c02-b72f-d898e6d1ea5d,Career Starter Stores,Aldi,SA44QN,Other,[],"['Communication', 'Teamwork', 'Problem Solving...",[],['Retailer qualification'],[],NaN,[],[],12.02,Unknown,Unknown
6,39ba0c82-a9cb-46f6-9e14-8bde7b6a83ea,Pharmacy Colleague,ASDA,WN86LU,Healthcare,[],[],[],[],[],NaN,NaN,[],13.10,Unknown,Unknown
7,f76d0f49-85ff-496f-8c59-04e9d13f03e0,Store Manager,Aldi,WA37PG,Retail,[],"['Leadership', 'Teamwork', 'Communication', 'P...",[],[],[],NaN,2 years of experience,[],60955.00,Unknown,Unknown
8,62207d2d-d4ed-4d09-a376-f75cce0b7494,Greggs Service Colleague Food Services,ASDA,SY84AL,Retail,[],[],[],[],[],NaN,NaN,[],13.10,Unknown,Unknown
9,de9579b9-1b14-482b-be01-dd5df18211af,CASS Oversight Analyst - 12mth FTC - M&G plc.,eFinancialCareers,Bath,Other,[],[],[],[],[],NaN,NaN,[],NaN,Unknown,Unknown


In [9]:
# SAVE FINAL LABOUR MARKET DATASET
FINAL_FILE = Path(
    r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\final_labour_market_dataset.csv"
)

final_df.to_csv(
    FINAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved successfully:")
print(FINAL_FILE)

print("\nFinal rows:", len(final_df))
print("Final columns:", len(final_df.columns))


Saved successfully:
C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\final_labour_market_dataset.csv

Final rows: 19207
Final columns: 29
